In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

In [ ]:
pipeline_id = ""
pipeline_name = ""
pipeline_run_id = ""
pipeline_trigger_id = ""
pipeline_trigger_time = ""
pipeline_trigger_type = ""
container = ""
datasubject = ""
classification = ""
sourcesystem = ""
sourceschema = ""
sourceschemaname = ""
sourcetablename = ""
tablename = ""
table_id = ""
ingest_channel = ""
ingest_partition = ""

In [ ]:
# Lấy lakehouse path tự động
lakehouse_path = spark.conf.get("spark.extraListeners", "").split(",")[0]

# Hoặc dùng absolute OneLake path trực tiếp
workspace_id = "cf008f6a-4054-4d95-9863-c4d514c799da"
lakehouse_id = "1a9e0a0b-eb37-491c-a088-d710c5849995"

file_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/brz/{datasubject}/{classification}/{sourcesystem}/{sourceschema}/{tablename}/{ingest_channel}/{ingest_partition}/"

output_table_name = f"sil_{datasubject}_{sourcesystem}_prd01_{tablename}"

target_file_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/sil/{datasubject}/{classification}/{sourcesystem}/{sourceschema}/{tablename}/"

print(f"Source path : {file_path}")
print(f"Target table: {output_table_name}")
print(f"Target path : {target_file_path}")

In [ ]:
data = spark.read.format("parquet") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("path", file_path) \
    .option("compression", "snappy") \
    .load()

# Chỉ ghi vào Delta file path — không cần default lakehouse context
data.write.mode("overwrite").format("delta").save(target_file_path)

print(f"Done: {data.count()} rows written to {target_file_path}")